# Physics-Informed PPO Relaxation — generalising over 4-cell meshes

Train a single PPO agent to relax **any** 4-cell mesh by moving vertices, reaching or
approaching the bilevel-optimised energy — in far fewer steps, and **generalising to
meshes unseen during training**.

## Architecture
- **GNN**: MeshGraphNet (edge→node message passing, no equivariance constraint).
- **Action**: per-vertex displacement (factorised Gaussian policy), mean guided by `-∇E`.
- **Action projection**: clip displacements so no edge falls below `min_dist_T1`,
  no vertex leaves the domain, and no vertex crosses an edge.
- **T1 transitions**: applied after projection iff they lower the energy.
- **Angles**: updated by gradient descent each step (like the bilevel inner loop).

## Generalisation
All 4-cell meshes share the same topology (6 verts, 18 half-edges, 4 faces, 9 edges),
so a batch of different meshes can be vmapped directly. Each env carries its **own**
mesh params (`vp/ep/fp`); each training iteration samples `N_ENVS` distinct meshes
from a dataset of `N_TRAIN`. Reward = normalised energy reduction + area-shape terms.
No bilevel baseline is needed during training — it is only computed at evaluation.

## Pipeline (run cells in order)
1. **Setup** — constants, helpers, shared-topology dims.
2. **Dataset** — generate/load `N_TRAIN` train + `N_TEST` held-out test meshes.
3. **MeshGraphNet** — policy/value network.
4. **Environment** — vectorised JAX env, each env relaxes its own mesh.
5. **PPO training** — GPU-batched rollout + update over sampled meshes; saves best model.
6. **Evaluation** — load model, relax 5 held-out test meshes, compare to bilevel.

In [ ]:
import os, sys, math
sys.path.insert(0, '/home/mcorbeau/Bureau/vertax/src/vertax/mitotic_spindle')
os.environ.setdefault('XLA_PYTHON_CLIENT_PREALLOCATE', 'false')

import numpy as np
import jax
import jax.numpy as jnp
from jax import vmap, jit, grad
import optax
import matplotlib.pyplot as plt
from functools import partial
from pathlib import Path
from topo_patched import update_T1_bounded_safe

from vertax import BoundedMesh
from vertax.topo import update_T1_bounded
from energy_adaptive import energy_adaptive
from boundedbop_patched import BoundedBilevelOptimizer

# ── Constants ─────────────────────────────────────────────────────────────────
N_CELLS     = 4
MIN_DIST_T1 = 0.01
MAX_STEPS   = 50
N_ENVS      = 256
ANG_MIN     = 0.017
ANG_MAX     = float(jnp.pi / 2 - 0.01)
POS_MIN     = -0.5
POS_MAX     = 2.5

# MeshGraphNet
H_DIM    = 128
N_LAYERS = 3

print('JAX devices:', jax.devices())


def mesh_energy(mesh):
    return float(energy_adaptive(
        jnp.array(mesh.vertices), jnp.array(mesh.angles), jnp.array(mesh.edges),
        jnp.array(mesh.faces), None, None, None,
        jnp.array(mesh.vertices_params), jnp.array(mesh.edges_params),
        jnp.array(mesh.faces_params),
    ))


def make_random_mesh(seed):
    """Unrelaxed 4-cell mesh with random tensions (deterministic in seed)."""
    mesh = BoundedMesh.from_random_seeds(
        nb_seeds=N_CELLS, width=math.sqrt(N_CELLS), height=math.sqrt(N_CELLS), random_key=seed
    )
    mesh.vertices_params = jnp.zeros(mesh.nb_vertices)
    mesh.edges_params    = jax.random.uniform(jax.random.PRNGKey(seed), (mesh.nb_edges,)) * 20 - 10
    mesh.faces_params    = jnp.full(mesh.nb_faces, 1.0)
    return mesh


# ── Shared topology dims (all 4-cell meshes: 6 verts, 18 he, 4 faces, 9 edges) ─
_ref    = make_random_mesh(0)
N_VERTS = _ref.nb_vertices
N_HE    = len(np.array(_ref.edges))
print(f'4-cell topology — n_verts={N_VERTS}, n_he={N_HE}, n_faces={_ref.nb_faces}, n_edges={_ref.nb_edges}')
print(f'Position bounds: [{POS_MIN}, {POS_MAX}], MAX_STEPS={MAX_STEPS}, N_ENVS={N_ENVS}')


JAX devices: [CudaDevice(id=0)]
4-cell topology — n_verts=6, n_he=18, n_faces=4, n_edges=9
Position bounds: [-0.5, 2.5], MAX_STEPS=50, N_ENVS=256


## 2. Dataset of 4-cell meshes

Generates (or loads) `N_TRAIN` training meshes + `N_TEST` held-out test meshes, each from a distinct random seed. All share the same topology, so their arrays can be stacked into `(N, ...)` tensors (`DS_X`, `DS_ANG`, `DS_HE`, `DS_FACE`, `DS_VP`, `DS_EP`, `DS_FP`) and sampled by `vmap` during training.

The dataset is cached to `ppo_dataset_4cells.pkl` (git-ignored). Test seeds (`N_TRAIN … N_TRAIN+N_TEST`) are **never seen during training** and are used only for the final generalisation evaluation.

In [2]:
# ── Dataset of 4-cell meshes (shared topology: 6 verts, 18 he, 4 faces, 9 edges) ──
import pickle
N_TRAIN      = 5000
N_TEST       = 5
DATASET_PATH = Path('/home/mcorbeau/Bureau/vertax/src/vertax/mitotic_spindle/ppo_dataset_4cells.pkl')

def mesh_arrays(seed):
    m = make_random_mesh(seed)
    return dict(
        X    = np.array(m.vertices,        np.float32),
        ang  = np.clip(np.array(m.angles,  np.float32), ANG_MIN, ANG_MAX),
        he   = np.array(m.edges,           np.int32),
        face = np.array(m.faces,           np.int32),
        vp   = np.array(m.vertices_params, np.float32),
        ep   = np.array(m.edges_params,    np.float32),
        fp   = np.array(m.faces_params,    np.float32),
    )

if DATASET_PATH.exists():
    with open(DATASET_PATH, 'rb') as f:
        ds = pickle.load(f)
    print(f'Loaded dataset from {DATASET_PATH}')
else:
    print(f'Generating {N_TRAIN} train + {N_TEST} test meshes...')
    train = [mesh_arrays(s) for s in range(N_TRAIN)]
    test  = [mesh_arrays(s) for s in range(N_TRAIN, N_TRAIN + N_TEST)]
    ds = {'train': train, 'test': test}
    with open(DATASET_PATH, 'wb') as f:
        pickle.dump(ds, f)
    print(f'Saved dataset to {DATASET_PATH}')

# Stack train arrays → (N_TRAIN, ...) for vmapped sampling
def stack_field(items, k):
    return jnp.array(np.stack([it[k] for it in items]))

DS_X    = stack_field(ds['train'], 'X')
DS_ANG  = stack_field(ds['train'], 'ang')
DS_HE   = stack_field(ds['train'], 'he')
DS_FACE = stack_field(ds['train'], 'face')
DS_VP   = stack_field(ds['train'], 'vp')
DS_EP   = stack_field(ds['train'], 'ep')
DS_FP   = stack_field(ds['train'], 'fp')
N_TRAIN_ACTUAL = DS_X.shape[0]
print(f'Train stacks: X={DS_X.shape}, he={DS_HE.shape}')


Loaded dataset from /home/mcorbeau/Bureau/vertax/src/vertax/mitotic_spindle/ppo_dataset_4cells.pkl
Train stacks: X=(5000, 6, 2), he=(5000, 18, 8)


## 3. MeshGraphNet policy

The policy network: a **MeshGraphNet** (edge↔node message passing) that outputs, per vertex, the mean and log-std of a Gaussian over the displacement, plus a state value (critic).

**Architecture:**
- Node-feature encoder (position, energy gradient, degree) and edge-feature encoder (relative vector, distance, tension).
- `N_LAYERS` message-passing blocks with residual connections.
- Heads: `alpha_head` (per-vertex step size, via softplus), `resid_head` (learned correction), `log_std_head` (exploration), `value_head` (critic).

**Physics-guided action:** the policy mean is `mean = -grad_En · alpha + residual`, i.e. an **energy gradient-descent step** (`-grad_En`, normalised gradient) scaled by a learned step size, plus a learned residual correction. The network therefore starts close to a useful descent step from episode 1.

`ACTION_SCALE` is deliberately small (`0.003`) so steps are ~10× smaller than a typical edge length → a vertex cannot jump across an edge in one step (like the bilevel with its tiny learning rate).

In [3]:
# ── MeshGraphNet (with LayerNorm, as in Pfaff et al. 2021) ────────────────────
#
# Action mean = -grad_En * alpha_per_vertex + residual
#   grad_En : NORMALISED gradient (unit norm)
# LayerNorm after each encoder and each message-passing MLP (not on output heads).

NODE_DIM     = 5
EDGE_DIM     = 4
ACTION_SCALE = 0.003
ACTION_CAP   = 5.0 * ACTION_SCALE


def mlp_init(key, sizes, small_last=False):
    params = []
    for i, (n_in, n_out) in enumerate(zip(sizes[:-1], sizes[1:])):
        key, sk = jax.random.split(key)
        is_last = (i == len(sizes) - 2)
        scale   = 1e-3 if (is_last and small_last) else math.sqrt(2.0 / n_in)
        W = jax.random.normal(sk, (n_in, n_out)) * scale
        b = jnp.zeros(n_out)
        params.append((W, b))
    return params


def mlp(params, x):
    for i, (W, b) in enumerate(params):
        x = x @ W + b
        if i < len(params) - 1:
            x = jax.nn.relu(x)
    return x


def layernorm_init(dim):
    return (jnp.ones(dim), jnp.zeros(dim))   # (gamma, beta)


def layernorm_apply(ln, x):
    """LayerNorm over the last axis (feature dim)."""
    gamma, beta = ln
    mu  = jnp.mean(x, axis=-1, keepdims=True)
    var = jnp.var(x, axis=-1, keepdims=True)
    return gamma * (x - mu) / jnp.sqrt(var + 1e-6) + beta


def init_mgn(key, h_dim=H_DIM, n_layers=N_LAYERS):
    p = {}
    key, k = jax.random.split(key)
    p['node_enc']    = mlp_init(k, [NODE_DIM, h_dim, h_dim])
    p['node_enc_ln'] = layernorm_init(h_dim)
    key, k = jax.random.split(key)
    p['edge_enc']    = mlp_init(k, [EDGE_DIM, h_dim, h_dim])
    p['edge_enc_ln'] = layernorm_init(h_dim)
    p['layers'] = []
    for _ in range(n_layers):
        key, k1, k2 = jax.random.split(key, 3)
        p['layers'].append({
            'phi_e':    mlp_init(k1, [h_dim*3, h_dim, h_dim]),
            'phi_e_ln': layernorm_init(h_dim),
            'phi_v':    mlp_init(k2, [h_dim*2, h_dim, h_dim]),
            'phi_v_ln': layernorm_init(h_dim),
        })
    key, k1, k2, k3 = jax.random.split(key, 4)
    p['alpha_head']   = mlp_init(k1, [h_dim, h_dim, 2], small_last=True)
    p['resid_head']   = mlp_init(k2, [h_dim, h_dim, 2], small_last=True)
    p['log_std_head'] = mlp_init(k3, [h_dim, h_dim, 2], small_last=True)
    key, k = jax.random.split(key)
    p['value_head'] = mlp_init(k, [h_dim, h_dim, 1])
    return p


def mgn_forward(p, X, srcs, tgts, tensions, grad_E, degree):
    n_verts = X.shape[0]
    g_norm  = jnp.linalg.norm(grad_E) + 1e-8
    grad_En = grad_E / g_norm

    node_feat = jnp.concatenate([X, grad_En, degree[:, None]], axis=1)
    H = vmap(lambda f: mlp(p['node_enc'], f))(node_feat)
    H = layernorm_apply(p['node_enc_ln'], H)

    dX        = X[srcs] - X[tgts]
    dist      = jnp.linalg.norm(dX, axis=1, keepdims=True)
    edge_feat = jnp.concatenate([dX, dist, tensions[:, None]], axis=1)
    E_feat    = vmap(lambda f: mlp(p['edge_enc'], f))(edge_feat)
    E_feat    = layernorm_apply(p['edge_enc_ln'], E_feat)

    for layer in p['layers']:
        msg_in = jnp.concatenate([H[srcs], H[tgts], E_feat], axis=1)
        msg    = vmap(lambda x: mlp(layer['phi_e'], x))(msg_in)
        E_feat = layernorm_apply(layer['phi_e_ln'], E_feat + msg)
        agg    = jnp.zeros((n_verts, H_DIM)).at[tgts].add(E_feat)
        upd    = vmap(lambda x: mlp(layer['phi_v'], x))(jnp.concatenate([H, agg], axis=1))
        H      = layernorm_apply(layer['phi_v_ln'], H + upd)

    alpha    = jax.nn.softplus(vmap(lambda h: mlp(p['alpha_head'], h))(H)) * ACTION_SCALE
    residual = vmap(lambda h: mlp(p['resid_head'], h))(H)
    mean     = -grad_En * alpha + residual * ACTION_SCALE
    mean     = jnp.clip(mean, -ACTION_CAP, ACTION_CAP)
    log_std  = jnp.clip(vmap(lambda h: mlp(p['log_std_head'], h))(H), -4.0, 0.0)
    value    = jnp.mean(vmap(lambda h: mlp(p['value_head'], h))(H))
    return mean, log_std, value


# Smoke test on an example mesh
key_mgn    = jax.random.PRNGKey(0)
mgn_params = init_mgn(key_mgn)

_m        = make_random_mesh(0)
X_test    = jnp.array(_m.vertices)
he_np     = np.array(_m.edges)
he_even   = he_np[::2]
srcs_test = he_even[:, 3] + he_even[:, 5] - 2
tgts_test = he_even[:, 4] + he_even[:, 6] - 3
valid     = (srcs_test >= 0) & (tgts_test >= 0)
srcs_test = jnp.array(srcs_test[valid]); tgts_test = jnp.array(tgts_test[valid])
tens_test = jnp.ones(int(valid.sum()))
deg_test  = jnp.ones(_m.nb_vertices) * 5.0
ang_t  = jnp.array(_m.angles); he_t = jnp.array(_m.edges)
face_t = jnp.array(_m.faces);  vp_t = jnp.array(_m.vertices_params)
ep_t   = jnp.array(_m.edges_params); fp_t = jnp.array(_m.faces_params)
gE_test = jax.grad(lambda Xg: energy_adaptive(
    Xg, ang_t, he_t, face_t, None, None, None, vp_t, ep_t, fp_t
))(X_test)

mean_t, lgs_t, val_t = mgn_forward(mgn_params, X_test, srcs_test, tgts_test, tens_test, gE_test, deg_test)
print(f'MeshGraphNet OK (LayerNorm) — |mean| max={float(jnp.max(jnp.abs(mean_t))):.2e}  '
      f'(ACTION_SCALE={ACTION_SCALE})')


MeshGraphNet OK (LayerNorm) — |mean| max=1.20e-03  (ACTION_SCALE=0.003)


## 4. Vectorised RL environment

Defines the environment dynamics, entirely in JAX (vmappable over the 256 envs). Each env carries its **own** mesh (`X, ang, he, face`) and params (`vp, ep, fp`), so the policy learns to relax any mesh, not a single one.

**Key functions:**
- `build_graph_jax`: extracts `(srcs, tgts, tensions, degree, mask)` from the half-edge table in pure JAX (boundary ghost edges are masked out).
- `energy_of` / `grad_energy_of` / `grad_energy_ang`: energy and its gradients w.r.t. positions and angles, all taking per-mesh `vp/ep/fp`.
- `cell_areas`, `area_dispersion`, `min_area`, `neg_area`: area metrics (the last detects **inverted** cells = negative signed area).
- `project_action`: **geometric safeguard** — bounds the action so (1) vertices stay in `[POS_MIN, POS_MAX]`, (2) no edge falls below `min_dist_T1`, (3) no vertex crosses an edge.
- `env_step(..., vp, ep, fp, E0, key)`: one full step — PPO action on `X` (guided by `-grad_E`), gradient step on the angles, conditional T1. Reward normalised by the mesh's own initial energy `E0`.
- `reset_envs(n, key)`: samples `n` meshes from the dataset and returns their stacked arrays + per-mesh `E0`.

**Reward** = normalised energy reduction − area-dispersion penalty + min-area bonus − strong inverted-area penalty, clamped to `[-1, 1]`.

In [4]:
# ── Vectorised JAX environment (generalises over a dataset of meshes) ─────────
#
# Each env now carries its OWN mesh params (vp/ep/fp) and topology (he/face),
# so the policy is trained to relax ANY 4-cell mesh, not a single one.
# All 4-cell meshes share shapes (6 verts, 18 he, 4 faces, 9 edges) → vmap works.

from vertax.geo import get_area_bounded

LOG_STD_MIN = -4.0
LOG_STD_MAX = -1.5
REWARD_CLIP = 1.0
K_AREA_VAR  = 5.0
K_AREA_MIN  = 2.0
K_NEG_AREA  = 50.0
ANG_LR      = 1e-3


def build_graph_jax(he, ep):
    he_even = he[::2]
    src_all = he_even[:, 3] + he_even[:, 5] - 2
    tgt_all = he_even[:, 4] + he_even[:, 6] - 3
    valid   = (src_all >= 0) & (tgt_all >= 0)
    mask    = valid.astype(jnp.float32)
    srcs = jnp.where(valid, src_all, 0)
    tgts = jnp.where(valid, tgt_all, 0)
    tensions = (jax.nn.sigmoid(ep) + 1) * mask
    deg = jnp.zeros(N_VERTS).at[srcs].add(mask).at[tgts].add(mask)
    return srcs, tgts, tensions, deg, mask


def energy_of(X, ang, he, face, vp, ep, fp):
    return energy_adaptive(X, ang, he, face, None, None, None, vp, ep, fp)


def grad_energy_of(X, ang, he, face, vp, ep, fp):
    g = jax.grad(lambda Xg: energy_of(Xg, ang, he, face, vp, ep, fp))(X)
    return jnp.nan_to_num(g, nan=0.0, posinf=0.0, neginf=0.0)


def grad_energy_ang(X, ang, he, face, vp, ep, fp):
    g = jax.grad(lambda a: energy_of(X, a, he, face, vp, ep, fp))(ang)
    return jnp.nan_to_num(g, nan=0.0, posinf=0.0, neginf=0.0)


def cell_areas(X, ang, he, face, fp):
    vt_full  = jnp.vstack([jnp.array([[0.0, 0.0], [1.0, 1.0]]), X])
    ang_full = jnp.repeat(ang, 2)
    faces    = jnp.arange(face.shape[0])
    return vmap(lambda f: get_area_bounded(f, vt_full, ang_full, he, face))(faces)


def area_dispersion(X, ang, he, face, fp):
    areas  = cell_areas(X, ang, he, face, fp)
    target = fp[0]
    return jnp.mean(((areas - target) / (target + 1e-8)) ** 2)


def min_area(X, ang, he, face, fp):
    areas  = cell_areas(X, ang, he, face, fp)
    return jnp.min(areas) / (fp[0] + 1e-8)


def neg_area(X, ang, he, face, fp):
    areas = cell_areas(X, ang, he, face, fp)
    return jnp.sum(jax.nn.relu(-areas)) / (fp[0] + 1e-8)


def get_edge_lengths_masked(X, srcs, tgts, mask):
    L = jnp.linalg.norm(X[srcs] - X[tgts], axis=1)
    return jnp.where(mask > 0, L, 1e3)


def _crossing_alpha(p, dp, a, da, b, db):
    def cross2d(u, v):
        return u[0] * v[1] - u[1] * v[0]
    B  = b - a;   dB = db - da
    P  = p - a;   dP = dp - da
    c0 = cross2d(B,  P)
    c1 = cross2d(B, dP) + cross2d(dB, P)
    c2 = cross2d(dB, dP)
    eps = 1e-7
    disc      = c1 * c1 - 4.0 * c2 * c0
    disc_safe = jnp.maximum(disc, 0.0)
    alpha1 = jnp.where(jnp.abs(c2) > eps,
                       (-c1 - jnp.sqrt(disc_safe)) / (2.0 * c2 + eps),
                       jnp.where(jnp.abs(c1) > eps, -c0 / c1, 1.0))
    alpha2 = jnp.where(jnp.abs(c2) > eps,
                       (-c1 + jnp.sqrt(disc_safe)) / (2.0 * c2 + eps), 1.0)
    def on_segment(alpha):
        pt = p + alpha * dp;  at = a + alpha * da;  bt = b + alpha * db
        seg  = bt - at
        proj = jnp.dot(pt - at, seg) / (jnp.dot(seg, seg) + eps)
        return (proj >= 0.0) & (proj <= 1.0)
    def safe_alpha(alpha):
        ok = (alpha > eps) & (alpha <= 1.0) & (disc >= 0.0) & on_segment(alpha)
        return jnp.where(ok, alpha * (1.0 - eps), 1.0)
    return jnp.minimum(safe_alpha(alpha1), safe_alpha(alpha2))


def project_action(X, action, srcs, tgts, mask, min_dist=MIN_DIST_T1):
    action = jnp.clip(X + action, POS_MIN, POS_MAX) - X
    X_new       = X + action
    new_lengths = get_edge_lengths_masked(X_new, srcs, tgts, mask)
    old_lengths = get_edge_lengths_masked(X,     srcs, tgts, mask)
    violated    = new_lengths < min_dist
    safe_len    = jnp.where(
        violated,
        jnp.clip((old_lengths - min_dist) / (old_lengths - new_lengths + 1e-8), 0.0, 1.0),
        1.0
    )
    n_e = srcs.shape[0]
    def alpha_for_pair(v_idx, e_idx):
        s = srcs[e_idx]; t = tgts[e_idx]; m = mask[e_idx]
        is_endpoint = (v_idx == s) | (v_idx == t)
        alpha = _crossing_alpha(X[v_idx], action[v_idx],
                                X[s], action[s], X[t], action[t])
        return jnp.where(m > 0, jnp.where(is_endpoint, 1.0, alpha), 1.0)
    vi = jnp.arange(N_VERTS)
    ei = jnp.arange(n_e)
    cross_alphas = vmap(lambda v: vmap(lambda e: alpha_for_pair(v, e))(ei))(vi)
    alpha = jnp.minimum(jnp.min(safe_len), jnp.min(cross_alphas))
    return X + alpha * action, alpha


def mgn_forward_masked(p, X, srcs, tgts, tensions, grad_E, degree, edge_mask):
    """Same as mgn_forward but zeroes out ghost (invalid) edges; with LayerNorm."""
    n_verts = X.shape[0]
    g_norm  = jnp.linalg.norm(grad_E) + 1e-8
    grad_En = grad_E / g_norm

    node_feat = jnp.concatenate([X, grad_En, degree[:, None]], axis=1)
    H = vmap(lambda f: mlp(p['node_enc'], f))(node_feat)
    H = layernorm_apply(p['node_enc_ln'], H)

    dX        = X[srcs] - X[tgts]
    dist      = jnp.linalg.norm(dX, axis=1, keepdims=True)
    edge_feat = jnp.concatenate([dX, dist, tensions[:, None]], axis=1)
    E_feat    = vmap(lambda f: mlp(p['edge_enc'], f))(edge_feat)
    E_feat    = layernorm_apply(p['edge_enc_ln'], E_feat) * edge_mask[:, None]

    for layer in p['layers']:
        msg_in = jnp.concatenate([H[srcs], H[tgts], E_feat], axis=1)
        msg    = vmap(lambda x: mlp(layer['phi_e'], x))(msg_in) * edge_mask[:, None]
        E_feat = layernorm_apply(layer['phi_e_ln'], E_feat + msg) * edge_mask[:, None]
        agg    = jnp.zeros((n_verts, H_DIM)).at[tgts].add(E_feat)
        upd    = vmap(lambda x: mlp(layer['phi_v'], x))(jnp.concatenate([H, agg], axis=1))
        H      = layernorm_apply(layer['phi_v_ln'], H + upd)

    alpha    = jax.nn.softplus(vmap(lambda h: mlp(p['alpha_head'], h))(H)) * ACTION_SCALE
    residual = vmap(lambda h: mlp(p['resid_head'], h))(H)
    mean     = -grad_En * alpha + residual * ACTION_SCALE
    mean     = jnp.clip(mean, -ACTION_CAP, ACTION_CAP)
    log_std  = jnp.clip(vmap(lambda h: mlp(p['log_std_head'], h))(H), LOG_STD_MIN, LOG_STD_MAX)
    value    = jnp.mean(vmap(lambda h: mlp(p['value_head'], h))(H))
    return mean, log_std, value


@jit
def env_step(params, X, ang, he, face, vp, ep, fp, E0, key):
    """One env step for a SINGLE mesh. E0 = that mesh's initial energy (reward norm)."""
    srcs, tgts, tensions, degree, mask = build_graph_jax(he, ep)
    grad_E = grad_energy_of(X, ang, he, face, vp, ep, fp)

    mean, log_std, value = mgn_forward_masked(params, X, srcs, tgts, tensions, grad_E, degree, mask)

    noise   = jax.random.normal(key, mean.shape)
    action  = mean + jnp.exp(log_std) * noise * ACTION_SCALE
    logp    = -0.5 * jnp.sum(((action - mean) / (jnp.exp(log_std) * ACTION_SCALE + 1e-8)) ** 2
                             + 2 * (log_std + jnp.log(ACTION_SCALE)) + jnp.log(2 * jnp.pi))

    E_before  = energy_of(X, ang, he, face, vp, ep, fp)
    ap_before = area_dispersion(X, ang, he, face, fp)
    ma_before = min_area(X, ang, he, face, fp)

    X_new, _ = project_action(X, action, srcs, tgts, mask)
    g_ang    = grad_energy_ang(X_new, ang, he, face, vp, ep, fp)
    ang_new  = jnp.clip(ang - ANG_LR * g_ang, ANG_MIN, ANG_MAX)

    vt1, at1, ht1, ft1 = update_T1_bounded_safe(
        X_new, ang_new, he, face, vp, ep, fp, energy_adaptive, MIN_DIST_T1
    )
    E_no_t1 = energy_of(X_new, ang_new, he, face, vp, ep, fp)
    E_t1    = energy_of(vt1, at1, ht1, ft1, vp, ep, fp)
    use_t1  = E_t1 < E_no_t1
    X_out   = jnp.where(use_t1, vt1, X_new)
    ang_out = jnp.where(use_t1, at1, ang_new)
    he_out  = jnp.where(use_t1, ht1, he)
    face_out= jnp.where(use_t1, ft1, face)

    E_new    = jnp.nan_to_num(energy_of(X_out, ang_out, he_out, face_out, vp, ep, fp), nan=1e12, posinf=1e12)
    ap_after = jnp.nan_to_num(area_dispersion(X_out, ang_out, he_out, face_out, fp), nan=1e3, posinf=1e3)
    ma_after = jnp.nan_to_num(min_area(X_out, ang_out, he_out, face_out, fp), nan=-1e3, neginf=-1e3)
    na_after = jnp.nan_to_num(neg_area(X_out, ang_out, he_out, face_out, fp), nan=1e3, posinf=1e3)

    r_energy = (E_before - E_new) / (jnp.abs(E0) + 1e-8)
    r_var    = -(ap_after - ap_before)
    r_min    =  (ma_after - ma_before)
    r_neg    = -na_after
    reward   = jnp.clip(r_energy + K_AREA_VAR * r_var + K_AREA_MIN * r_min + K_NEG_AREA * r_neg,
                        -REWARD_CLIP, REWARD_CLIP)

    return X_out, ang_out, he_out, face_out, action, logp, value, reward, E_new


# vmap over envs: params shared (None), everything else per-env (0)
env_step_batch = jit(vmap(env_step, in_axes=(None, 0, 0, 0, 0, 0, 0, 0, 0, 0)))


def reset_envs(n, key):
    """Sample n meshes from the training dataset and return their stacked arrays + E0."""
    idx  = jax.random.randint(key, (n,), 0, N_TRAIN_ACTUAL)
    X    = DS_X[idx]; ang = DS_ANG[idx]; he = DS_HE[idx]; fc = DS_FACE[idx]
    vp   = DS_VP[idx]; ep = DS_EP[idx];  fp = DS_FP[idx]
    E0 = vmap(energy_of)(X, ang, he, fc, vp, ep, fp)
    return X, ang, he, fc, vp, ep, fp, E0


# Smoke test
Xb, angb, heb, fcb, vpb, epb, fpb, E0b = reset_envs(4, jax.random.PRNGKey(0))
keys = jax.random.split(jax.random.PRNGKey(1), 4)
out  = env_step_batch(mgn_params, Xb, angb, heb, fcb, vpb, epb, fpb, E0b, keys)
Xn, angn, hen, fcn, act, lp, val, rew, En = out
print(f'Vectorised env OK (LayerNorm) — X {Xn.shape}, rewards {rew.shape}')
print(f'  E0[:4]={np.array(E0b)}')
print(f'  rewards[:4]={np.array(rew)}')


Vectorised env OK (LayerNorm) — X (4, 6, 2), rewards (4,)
  E0[:4]=[610.35944 657.1252  591.1811  663.2105 ]
  rewards[:4]=[0.04211953 0.02884232 0.07200952 0.01902476]


## 5. PPO training (generalising over the dataset)

Full PPO loop in JAX, **256 environments in parallel**. Each iteration samples 256 **distinct** meshes from the training dataset, so the policy is optimised across the whole distribution of 4-cell meshes.

**Components:**
- `rollout`: samples `N_ENVS` meshes, rolls out `MAX_STEPS` via `lax.scan`. Per-step arrays (`X/ang/action/logp/value/reward/E`) are stored; per-env constants (`he/face/vp/ep/fp/E0`) are returned separately (they don't change during an episode → memory saved).
- `compute_gae`: Generalized Advantage Estimation over `(T, N_ENVS)`, fixed horizon.
- `ppo_update`: clipped PPO loss (surrogate + value + entropy). Per-env constants are tiled over the `T` steps so each sample sees its own mesh params.
- **Cosine annealing** of the learning rate.
- **Checkpoint** of the best model (highest mean energy reduction `ΔE%`) to `ppo_mgn_params.pkl`.

**Logs** (every `LOG_EVERY` iters): mean return, mean ΔE% over the batch, lr, loss. No bilevel needed here.

In [5]:
# ── PPO (vectorised, generalises over the mesh dataset) ───────────────────────
# Each iteration samples N_ENVS distinct meshes from the dataset. he/face/vp/ep/fp
# are constant per episode (stored once per env, not per step) to save memory.
import pickle
from pathlib import Path

N_ITERS      = 500
LR_PEAK      = 3e-4
LR_MIN       = 1e-6
GAMMA        = 0.99
LAM          = 0.95
CLIP_EPS     = 0.2
N_EPOCHS_PPO = 4
ENT_COEF     = 1e-3
VF_COEF      = 0.5
RET_CLIP     = 50.0
LOG_EVERY    = 10
MODEL_PATH   = Path('/home/mcorbeau/Bureau/vertax/src/vertax/mitotic_spindle/ppo_mgn_params.pkl')

lr_schedule = optax.cosine_decay_schedule(
    init_value=LR_PEAK, decay_steps=N_ITERS * N_EPOCHS_PPO, alpha=LR_MIN / LR_PEAK
)
optimizer_ppo = optax.chain(optax.clip_by_global_norm(0.5), optax.adam(learning_rate=lr_schedule))
mgn_params = init_mgn(jax.random.PRNGKey(0))
opt_state  = optimizer_ppo.init(mgn_params)


def action_log_prob(action, mean, log_std):
    std = jnp.exp(log_std) * ACTION_SCALE
    return -0.5 * jnp.sum(((action - mean) / (std + 1e-8)) ** 2
                          + 2 * jnp.log(std + 1e-8) + jnp.log(2 * jnp.pi))


@jit
def rollout(params, key):
    """Sample N_ENVS meshes, roll out MAX_STEPS. Per-step: X/ang/action/logp/value/reward/E.
    Per-env constants (he/face/vp/ep/fp/E0) returned separately."""
    rk, sk = jax.random.split(key)
    X, ang, he, fc, vp, ep, fp, E0 = reset_envs(N_ENVS, rk)

    def scan_step(carry, step_key):
        X, ang = carry
        keys = jax.random.split(step_key, N_ENVS)
        X2, ang2, _, _, action, logp, value, reward, E_new = env_step_batch(
            params, X, ang, he, fc, vp, ep, fp, E0, keys
        )
        out = dict(X=X, ang=ang, action=action, logp=logp, value=value, reward=reward, E=E_new)
        return (X2, ang2), out

    step_keys = jax.random.split(sk, MAX_STEPS)
    _, traj = jax.lax.scan(scan_step, (X, ang), step_keys)
    consts = dict(he=he, face=fc, vp=vp, ep=ep, fp=fp, E0=E0)
    return traj, consts


def compute_gae(rewards, values, gamma=GAMMA, lam=LAM):
    T = rewards.shape[0]
    advs = jnp.zeros_like(rewards)
    def body(t, carry):
        gae_prev, advs = carry
        t_rev    = T - 1 - t
        next_val = jnp.where(t == 0, 0.0, values[jnp.minimum(t_rev + 1, T - 1)])
        delta    = rewards[t_rev] + gamma * next_val - values[t_rev]
        gae_now  = delta + gamma * lam * gae_prev
        advs     = advs.at[t_rev].set(gae_now)
        return (gae_now, advs)
    _, advs = jax.lax.fori_loop(0, T, body, (jnp.zeros(rewards.shape[1]), advs))
    return advs, advs + values


@jit
def ppo_update(params, opt_state, traj, consts, advs, returns):
    T, N = traj['reward'].shape
    flat = lambda a: a.reshape((T * N,) + a.shape[2:])
    # Per-env constants tiled over the T steps, then flattened
    tile = lambda a: jnp.broadcast_to(a[None], (T,) + a.shape).reshape((T * N,) + a.shape[1:])

    Xf, angf    = flat(traj['X']), flat(traj['ang'])
    actf, logpf = flat(traj['action']), flat(traj['logp'])
    advf, retf  = flat(advs), flat(returns)
    hef, facef  = tile(consts['he']), tile(consts['face'])
    vpf, epf, fpf = tile(consts['vp']), tile(consts['ep']), tile(consts['fp'])
    advf = (advf - advf.mean()) / (advf.std() + 1e-8)
    retf = jnp.clip(retf, -RET_CLIP, RET_CLIP)

    def loss_fn(p):
        def per_sample(X, ang, he, fc, vp, ep, fp, action, old_logp, adv, ret):
            srcs, tgts, tens, deg, mask = build_graph_jax(he, ep)
            gE = grad_energy_of(X, ang, he, fc, vp, ep, fp)
            mean, log_std, value = mgn_forward_masked(p, X, srcs, tgts, tens, gE, deg, mask)
            logp  = action_log_prob(action, mean, log_std)
            ratio = jnp.exp(jnp.clip(logp - old_logp, -10.0, 10.0))
            pg    = -jnp.minimum(ratio * adv, jnp.clip(ratio, 1-CLIP_EPS, 1+CLIP_EPS) * adv)
            vf    = (value - ret) ** 2
            ent   = jnp.sum(log_std + 0.5 * jnp.log(2 * jnp.pi * jnp.e))
            return pg + VF_COEF * vf - ENT_COEF * ent
        return jnp.mean(vmap(per_sample)(Xf, angf, hef, facef, vpf, epf, fpf, actf, logpf, advf, retf))

    loss, grads = jax.value_and_grad(loss_fn)(params)
    updates, opt_state = optimizer_ppo.update(grads, opt_state, params)
    params = optax.apply_updates(params, updates)
    return params, opt_state, loss


# ── Training loop ─────────────────────────────────────────────────────────────
key = jax.random.PRNGKey(0)
iter_returns, iter_dpct, iter_losses = [], [], []
best_metric = -1e9   # track best mean ΔE% for checkpointing
stop_training = False
update_count = 0

print(f'Vectorised PPO — generalising over {N_TRAIN_ACTUAL} meshes')
print(f'  {N_ITERS} iters × {N_ENVS} envs × {MAX_STEPS} steps, ACTION_SCALE={ACTION_SCALE}')
print(f'{"Iter":>6} {"meanRet":>9} {"meanΔE%":>9} {"lr":>9} {"loss":>10}')
print('-' * 50)

for it in range(N_ITERS):
    key, rk = jax.random.split(key)
    traj, consts = rollout(mgn_params, rk)
    advs, returns = compute_gae(traj['reward'], traj['value'])

    ep_loss = 0.0
    for _ in range(N_EPOCHS_PPO):
        mgn_params, opt_state, loss = ppo_update(mgn_params, opt_state, traj, consts, advs, returns)
        ep_loss = float(loss)
        update_count += 1
        if not np.isfinite(ep_loss):
            print(f'\n⚠️  NaN/Inf loss at iter {it+1} — stopping.')
            stop_training = True
            break
    if stop_training:
        break

    # Mean energy reduction across the batch (final E vs initial E0)
    final_E   = np.array(traj['E'][-1])
    E0        = np.array(consts['E0'])
    mean_dpct = float(np.mean((E0 - final_E) / (np.abs(E0) + 1e-8)) * 100)
    mean_ret  = float(jnp.sum(traj['reward'], axis=0).mean())
    iter_returns.append(mean_ret)
    iter_dpct.append(mean_dpct)
    iter_losses.append(ep_loss)

    if mean_dpct > best_metric:
        best_metric = mean_dpct
        with open(MODEL_PATH, 'wb') as f:
            pickle.dump({'params': mgn_params, 'best_dpct': best_metric, 'iter': it + 1}, f)

    if (it + 1) % LOG_EVERY == 0:
        lr_cur = float(lr_schedule(update_count))
        print(f'{it+1:>6d} {mean_ret:>9.3f} {mean_dpct:>8.1f}% {lr_cur:>9.2e} {ep_loss:>10.4f}')

if not stop_training:
    with open(MODEL_PATH, 'wb') as f:
        pickle.dump({'params': mgn_params, 'best_dpct': best_metric, 'iter': N_ITERS}, f)
    print(f'\nModel saved to {MODEL_PATH} (best mean ΔE% = {best_metric:.1f}%)')


Vectorised PPO — generalising over 5000 meshes
  500 iters × 256 envs × 50 steps, ACTION_SCALE=0.003
  Iter   meanRet   meanΔE%        lr       loss
--------------------------------------------------


KeyboardInterrupt: 

## 6. Evaluation on held-out test meshes

Loads the best saved model and evaluates **generalisation** on the `N_TEST` test meshes that were **never seen during training** (seeds `N_TRAIN … N_TRAIN+N_TEST`).

For each test mesh:
- deterministic PPO rollout (policy mean, no noise) for `MAX_STEPS`, with the per-step angle gradient update;
- bilevel optimisation of the same mesh as reference;
- the three meshes (initial / PPO / bilevel) are rendered side by side, coloured by tension (edges) and area (cells).

Output: a `N_TEST × 3` grid of meshes plus a table of energies (`E_init`, `E_PPO`, `E_bilevel`) per test mesh. This shows whether the single trained policy relaxes unseen meshes as well as the per-mesh bilevel optimiser.

### Aside — the nonlinear Conjugate Gradient relaxer (`relax_cg`)

The CG column relaxes a mesh by minimising the **same energy** as every other column
(`energy_adaptive`, which already contains the target-area term `K_AREAS·Σ(area−fp[0])²`),
at **fixed topology**. It is a *nonlinear* conjugate gradient — Polak–Ribière flavour —
with restarts, a zoom line search, and the same geometric projection PPO uses.

**Why not just gradient descent?**
Steepest descent (`d = -∇E`) zig-zags badly in elongated valleys: each step undoes part
of the previous one. CG fixes this by making successive search directions **conjugate**
(roughly: non-interfering with respect to the local curvature), so progress along the
valley is not destroyed at the next step. On a purely quadratic objective in `n`
variables it converges in at most `n` steps; our energy is not quadratic, hence
"nonlinear" CG with periodic restarts.

**One iteration (`step`), in order:**

1. **Gradient** `g = ∇E` at the current `(X, ang)`.
2. **Polak–Ribière coefficient**
   `β = ⟨g_k , g_k − g_{k-1}⟩ / ‖g_{k-1}‖²`.
   It measures how much of the *previous* direction to keep. The `g_k − g_{k-1}` term
   (the gradient *change*) is what injects curvature information without ever forming a
   Hessian.
3. **Restart** (set `β = 0`, i.e. fall back to `-g`) when either:
   - every `N_RESTART_CG` steps — conjugacy degrades on a non-quadratic objective, so we
     refresh it; or
   - `β ≤ 0` — this is the **PR+** safeguard `β⁺ = max(0, β)`: a negative `β` means the
     old direction is no longer useful, so we drop it.
4. **Search direction** `d_k = -g_k + β·d_{k-1}`
   (`β = 0` ⇒ steepest descent; `β > 0` ⇒ curve the search along the valley).
5. **Descent guard**: a line search is only valid if `d` goes downhill (`⟨d, g⟩ < 0`).
   If rounding / non-quadratic effects break that, fall back to `-g`.
6. **Zoom line search** along `d`: picks the step size satisfying the (strong) Wolfe
   conditions by re-evaluating `E` along the line. This is the part L-BFGS and CG share,
   and the part that demands a **smooth, stationary** energy (why T1 is kept *off* during
   the relaxation and applied once at the end).
7. **Geometric projection** (`project_action`, the PPO safeguard): scales the vertex step
   by `α ∈ [0, 1]` so no vertex crosses an edge and no edge collapses below `MIN_DIST_T1`.
   `α < 1` means the raw step was clipped.
8. **Restart on clip**: if the projection clipped the step (`α < 1`), the path actually
   taken no longer matches the direction CG built its conjugacy on, so we force a full
   restart (direction → `-g`, fresh line-search state) on the projected point. This is
   what removed the edge-crossing artefacts (e.g. seed 5001).

**Relation to L-BFGS.** Both use the same zoom line search, the same projection, and the
same restart-on-clip rule. The difference is memory: CG keeps **one** previous direction
(a single vector), L-BFGS keeps the last `m` gradient/step pairs to approximate the
inverse Hessian. CG is therefore lighter and more robust to the topology change at the
final T1 (less stale memory to invalidate), while L-BFGS can converge slightly deeper on
the smooth part. On these 6-vertex meshes they land at essentially the same energy.


In [ ]:
# ── Evaluation on 5 held-out test meshes (unseen during training) ─────────────
# Columns: Initial | PPO | CG | L-BFGS | Bilevel
import pickle
from pathlib import Path
from vertax import get_plot_mesh, EdgePlot, FacePlot, VertexPlot
import optax.tree_utils as otu

MODEL_PATH = Path('/home/mcorbeau/Bureau/vertax/src/vertax/mitotic_spindle/ppo_mgn_params.pkl')
if MODEL_PATH.exists():
    with open(MODEL_PATH, 'rb') as f:
        ckpt = pickle.load(f)
    mgn_params_eval = ckpt['params']
    print(f'Loaded model — best mean ΔE%={ckpt.get("best_dpct", float("nan")):.1f}%, iter={ckpt["iter"]}')
else:
    mgn_params_eval = mgn_params
    print('No saved model, using in-memory params.')

PLOT_KW = dict(
    edge_plot=EdgePlot.EDGE_PARAMETER, edge_parameters_name='tension',
    face_plot=FacePlot.AREA, face_parameters_name='area',
    vertex_plot=VertexPlot.INVISIBLE,
)

# ── Iteration budget for the classical optimisers (CG / L-BFGS) ───────────────
CLASSIC_ITERS = 200   # CG / L-BFGS converge well within this; topology kept fixed


def render_mesh(mesh, title):
    mc = BoundedMesh.copy_mesh(mesh)
    mc.angles = jnp.clip(jnp.array(mc.angles), ANG_MIN, ANG_MAX)
    fig_m, _ = get_plot_mesh(mc, title=title, **PLOT_KW)
    fig_m.canvas.draw()
    w, h = fig_m.canvas.get_width_height()
    img = np.frombuffer(fig_m.canvas.tostring_argb(), dtype=np.uint8).reshape(h, w, 4)
    img = np.roll(img, -1, axis=2)
    plt.close(fig_m)
    return img


def build_bounded_mesh(seed):
    """Rebuild the exact test mesh from its seed (same as make_random_mesh)."""
    return make_random_mesh(seed)


# ── Deterministic PPO eval step (fixed topology, per-mesh params) ─────────────
@jit
def eval_step_mesh(params, X, ang, he, face, vp, ep, fp):
    srcs, tgts, tens, deg, mask = build_graph_jax(he, ep)
    gE   = grad_energy_of(X, ang, he, face, vp, ep, fp)
    mean, _, _ = mgn_forward_masked(params, X, srcs, tgts, tens, gE, deg, mask)
    X_new, _ = project_action(X, mean, srcs, tgts, mask)
    g_ang    = grad_energy_ang(X_new, ang, he, face, vp, ep, fp)
    ang_new  = jnp.clip(ang - ANG_LR * g_ang, ANG_MIN, ANG_MAX)
    E_new    = jnp.nan_to_num(energy_of(X_new, ang_new, he, face, vp, ep, fp), nan=1e12, posinf=1e12)
    return X_new, ang_new, E_new


# ── Classical relaxation (CG / L-BFGS) — fixed topology, line-search solvers ───
#
# Both CG and L-BFGS require a *line search* that re-evaluates the loss along the
# descent direction. We optimise the pytree params=(X, ang) on the energy, with
# he/face/vp/ep/fp held fixed (smooth, stationary objective → assumptions hold).
# Angles are clipped to [ANG_MIN, ANG_MAX] after each step.
#
# optax 0.2.4 ships optax.lbfgs() (with built-in zoom line search) but no
# nonlinear-CG. We build Polak–Ribière CG by hand, sharing the same zoom line
# search, with a restart (beta -> 0) every N_RESTART steps and whenever the PR
# coefficient goes negative.

N_RESTART_CG = 10


def _relax_loss(params, he, face, vp, ep, fp):
    X, ang = params
    ang_c  = jnp.clip(ang, ANG_MIN, ANG_MAX)
    return jnp.nan_to_num(energy_of(X, ang_c, he, face, vp, ep, fp),
                          nan=1e12, posinf=1e12, neginf=1e12)


def relax_lbfgs(X0, ang0, he, face, vp, ep, fp, n_iter=CLASSIC_ITERS):
    """L-BFGS relaxation with optax's built-in zoom line search, topology fixed.

    Each X-step is passed through ``project_action`` (the same geometric safeguard
    PPO uses) so no vertex crosses an edge and no edge collapses below ``MIN_DIST_T1``.
    When the projection clips the step (alpha < 1), the L-BFGS curvature memory is
    no longer consistent with the path actually taken, so we **reset** the solver
    state onto the projected point (restores the stationarity assumption).
    """
    loss = lambda p: _relax_loss(p, he, face, vp, ep, fp)
    srcs, tgts, _, _, mask = build_graph_jax(he, ep)
    solver = optax.lbfgs()
    value_and_grad = optax.value_and_grad_from_state(loss)

    def step(carry, _):
        params, state = carry
        v, g = value_and_grad(params, state=state)
        updates, state = solver.update(g, state, params,
                                       value=v, grad=g, value_fn=loss)
        X, ang   = params
        dX, dang = updates
        Xp, alpha = project_action(X, dX, srcs, tgts, mask)   # geometric safeguard
        ang_new   = jnp.clip(ang + dang, ANG_MIN, ANG_MAX)
        new_params = (Xp, ang_new)
        # Reset curvature memory if the projection altered the step.
        state = jax.lax.cond(alpha < 1.0,
                             lambda: solver.init(new_params), lambda: state)
        return (new_params, state), None

    params0 = (X0, jnp.clip(ang0, ANG_MIN, ANG_MAX))
    state0  = solver.init(params0)
    (params, _), _ = jax.lax.scan(step, (params0, state0), None, length=n_iter)
    X, ang = params
    return X, jnp.clip(ang, ANG_MIN, ANG_MAX)


def relax_cg(X0, ang0, he, face, vp, ep, fp, n_iter=CLASSIC_ITERS):
    """Nonlinear conjugate gradient (Polak–Ribière + restart), shared zoom line search.

    Like ``relax_lbfgs``, each X-step is projected with ``project_action`` to keep the
    mesh geometrically valid (no edge crossing / collapse). When the projection clips
    the step (alpha < 1), we force a **full CG restart**: direction reset to steepest
    descent and a fresh line-search state on the projected point.
    """
    loss = lambda p: _relax_loss(p, he, face, vp, ep, fp)
    srcs, tgts, _, _, mask = build_graph_jax(he, ep)
    ls = optax.scale_by_zoom_linesearch(max_linesearch_steps=20)

    def step(carry, i):
        # Carry between iterations:
        #   params  = (X, ang)  current point
        #   g_prev  = gradient at the previous point   (for the Polak–Ribière ratio)
        #   d_prev  = previous conjugate direction      (the term that makes it "conjugate")
        #   ls_state= zoom line-search state
        params, g_prev, d_prev, ls_state = carry

        # (1) Gradient of the energy at the current point.
        #     g is the steepest-DESCENT direction is -g; CG improves on it below.
        v, g = jax.value_and_grad(loss)(params)

        # (2) Polak–Ribière coefficient beta_k.
        #     Plain gradient descent uses d = -g and zig-zags in narrow valleys.
        #     CG mixes in a fraction beta of the previous direction so successive
        #     directions are ~conjugate w.r.t. the local curvature → far less zig-zag.
        #         beta_PR = <g_k, g_k - g_{k-1}> / <g_{k-1}, g_{k-1}>
        #     (vdot here flattens the (X, ang) pytree into one big vector.)
        gg_prev = otu.tree_vdot(g_prev, g_prev) + 1e-12      # ||g_{k-1}||^2  (denominator)
        y       = otu.tree_sub(g, g_prev)                    # g_k - g_{k-1}  (gradient change)
        beta_pr = otu.tree_vdot(g, y) / gg_prev

        # (3) Restart = drop the memory and take a pure -g step (beta := 0). We restart:
        #       • every N_RESTART_CG steps  → classic periodic restart; on a non-quadratic
        #         objective the conjugacy degrades, so we refresh it regularly;
        #       • whenever beta_PR <= 0     → the auto-restart property of PR+ (beta_PR^+ =
        #         max(0, beta_PR)); a negative beta means the old direction is no longer
        #         helpful, so we discard it.
        do_restart = (i % N_RESTART_CG == 0) | (beta_pr <= 0.0)
        beta = jnp.where(do_restart, 0.0, beta_pr)

        # (4) New search direction:  d_k = -g_k + beta_k * d_{k-1}
        #     beta=0 → steepest descent; beta>0 → curve the search along the valley.
        d = otu.tree_add_scalar_mul(otu.tree_scalar_mul(-1.0, g), beta, d_prev)

        # (5) Descent-direction guard. A line search only makes sense if d actually goes
        #     downhill, i.e. <d, g> < 0. Rounding / non-quadratic effects can break this;
        #     if so, fall back to the always-valid steepest-descent direction -g.
        is_descent = otu.tree_vdot(d, g) < 0.0
        d = jax.lax.cond(is_descent, lambda: d, lambda: otu.tree_scalar_mul(-1.0, g))

        # (6) Zoom line search along d → chooses the step size satisfying the (strong)
        #     Wolfe conditions, re-evaluating the energy along d. `updates` = step * d.
        updates, ls_state = ls.update(
            updates=d, state=ls_state, params=params,
            value=v, grad=g, value_fn=loss)
        X, ang   = params
        dX, dang = updates

        # (7) Geometric safeguard: project the vertex step so no vertex crosses an edge
        #     and no edge collapses below MIN_DIST_T1. alpha in [0, 1] scales dX along the
        #     SAME direction (so it stays a descent step), alpha < 1 ⇒ the step was clipped.
        Xp, alpha = project_action(X, dX, srcs, tgts, mask)
        ang_new   = jnp.clip(ang + dang, ANG_MIN, ANG_MAX)
        new_params = (Xp, ang_new)

        # (8) If the projection clipped the step, the path actually taken no longer matches
        #     the direction CG built its conjugacy on → force a FULL restart next step:
        #     direction reset to -g and a fresh line-search state on the projected point.
        clipped  = alpha < 1.0
        d_out    = jax.lax.cond(clipped, lambda: otu.tree_scalar_mul(-1.0, g), lambda: d)
        ls_state = jax.lax.cond(clipped, lambda: ls.init(new_params), lambda: ls_state)
        return (new_params, g, d_out, ls_state), None

    params0 = (X0, jnp.clip(ang0, ANG_MIN, ANG_MAX))
    g0      = jax.grad(loss)(params0)
    d0      = otu.tree_scalar_mul(-1.0, g0)
    ls0     = ls.init(params0)
    (params, _, _, _), _ = jax.lax.scan(
        step, (params0, g0, d0, ls0), jnp.arange(n_iter))
    X, ang = params
    return X, jnp.clip(ang, ANG_MIN, ANG_MAX)


def apply_final_T1(X, ang, he, face, vp, ep, fp):
    """One T1 pass after the (fixed-topology) relaxation, accepted iff it lowers E."""
    vt1, at1, ht1, ft1 = update_T1_bounded_safe(
        X, ang, he, face, vp, ep, fp, energy_adaptive, MIN_DIST_T1)
    E_no = energy_of(X, ang, he, face, vp, ep, fp)
    E_t1 = energy_of(vt1, at1, ht1, ft1, vp, ep, fp)
    use  = E_t1 < E_no
    return (jnp.where(use, vt1, X), jnp.where(use, at1, ang),
            jnp.where(use, ht1, he), jnp.where(use, ft1, face))


relax_lbfgs_j = jit(relax_lbfgs)
relax_cg_j    = jit(relax_cg)
apply_final_T1_j = jit(apply_final_T1)


def repeat_ep_for_plot(mesh):
    """get_plot_mesh indexes edges_params per half-edge; repeat if stored per edge."""
    ep   = np.asarray(mesh.edges_params)
    n_he = len(np.asarray(mesh.edges))
    if ep.shape[0] != n_he:
        return np.repeat(ep, n_he // ep.shape[0])
    return ep


def make_result_mesh(m_init, X, ang, he, face):
    """Build a BoundedMesh for plotting from relaxed arrays."""
    mr = BoundedMesh.copy_mesh(m_init)
    mr.vertices     = np.array(X)
    mr.angles       = np.array(ang)
    mr.edges        = np.array(he)
    mr.faces        = np.array(face)
    mr.edges_params = repeat_ep_for_plot(m_init)
    return mr


test_seeds = list(range(N_TRAIN, N_TRAIN + N_TEST))
N_COLS = 5
fig, axes = plt.subplots(N_TEST, N_COLS, figsize=(5 * N_COLS, 5 * N_TEST), constrained_layout=True)
if N_TEST == 1:
    axes = axes[None, :]

hdr = f'\n{"Test mesh":<12} {"E_init":>10} {"E_PPO":>10} {"E_CG":>10} {"E_LBFGS":>10} {"E_bilevel":>10}'
print(hdr)
print('-' * len(hdr))

for row, seed in enumerate(test_seeds):
    # Initial mesh
    m_init = build_bounded_mesh(seed)
    E_init_t = mesh_energy(m_init)

    X0    = jnp.array(m_init.vertices)
    ang0  = jnp.clip(jnp.array(m_init.angles), ANG_MIN, ANG_MAX)
    he0   = jnp.array(m_init.edges)
    fc0   = jnp.array(m_init.faces)
    vp_e  = jnp.array(m_init.vertices_params)
    ep_e  = jnp.array(m_init.edges_params)
    fp_e  = jnp.array(m_init.faces_params)

    # ── PPO deterministic rollout ──
    X_e, ang_e = X0, ang0
    for _ in range(MAX_STEPS):
        X_e, ang_e, _ = eval_step_mesh(mgn_params_eval, X_e, ang_e, he0, fc0, vp_e, ep_e, fp_e)
    E_ppo_t = float(energy_of(X_e, ang_e, he0, fc0, vp_e, ep_e, fp_e))
    m_ppo = make_result_mesh(m_init, X_e, ang_e, he0, fc0)

    # ── CG relaxation (fixed topology) + final T1 ──
    Xc, angc = relax_cg_j(X0, ang0, he0, fc0, vp_e, ep_e, fp_e)
    Xc, angc, hec, fcc = apply_final_T1_j(Xc, angc, he0, fc0, vp_e, ep_e, fp_e)
    E_cg_t = float(energy_of(Xc, angc, hec, fcc, vp_e, ep_e, fp_e))
    m_cg = make_result_mesh(m_init, Xc, angc, hec, fcc)

    # ── L-BFGS relaxation (fixed topology) + final T1 ──
    Xl, angl = relax_lbfgs_j(X0, ang0, he0, fc0, vp_e, ep_e, fp_e)
    Xl, angl, hel, fcl = apply_final_T1_j(Xl, angl, he0, fc0, vp_e, ep_e, fp_e)
    E_lbfgs_t = float(energy_of(Xl, angl, hel, fcl, vp_e, ep_e, fp_e))
    m_lbfgs = make_result_mesh(m_init, Xl, angl, hel, fcl)

    # ── Bilevel baseline (Adam, T1 interleaved) ──
    m_bil = BoundedMesh.copy_mesh(m_init)
    opt_t = BoundedBilevelOptimizer()
    opt_t.loss_function_inner = energy_adaptive
    opt_t.inner_solver        = optax.adam(learning_rate=1e-4)
    opt_t.update_T1           = True
    opt_t._update_T1_func     = update_T1_bounded
    opt_t.min_dist_T1         = MIN_DIST_T1
    opt_t.max_nb_iterations   = 2000
    opt_t.tolerance           = 1e-6
    opt_t.patience            = 50
    opt_t.inner_optimization(m_bil)
    E_bil_t = mesh_energy(m_bil)

    print(f'seed {seed:<7d} {E_init_t:>10.2f} {E_ppo_t:>10.2f} {E_cg_t:>10.2f} {E_lbfgs_t:>10.2f} {E_bil_t:>10.2f}')

    for col, (m, ttl) in enumerate([
        (m_init,  f'Initial (E={E_init_t:.1f})'),
        (m_ppo,   f'PPO {MAX_STEPS} steps (E={E_ppo_t:.1f})'),
        (m_cg,    f'CG {CLASSIC_ITERS} it (E={E_cg_t:.1f})'),
        (m_lbfgs, f'L-BFGS {CLASSIC_ITERS} it (E={E_lbfgs_t:.1f})'),
        (m_bil,   f'Bilevel (E={E_bil_t:.1f})'),
    ]):
        try:
            axes[row, col].imshow(render_mesh(m, ttl))
        except Exception as e:
            axes[row, col].text(0.5, 0.5, f'render failed:\n{e}', ha='center', va='center', fontsize=7, wrap=True)
        axes[row, col].set_title(f'seed {seed} — {ttl}', fontsize=10)
        axes[row, col].axis('off')

fig.suptitle('Generalisation: 5 held-out test meshes (initial / PPO / CG / L-BFGS / bilevel)', fontsize=14)
plt.show()
